In [ ]:
import pandas as pd
import numpy as np
import re
from collections import Counter, defaultdict

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.neighbors import NearestNeighbors

In [ ]:
import pandas as pd
from google.colab import files
from sklearn.model_selection import train_test_split

# Upload:
# processed_icd10_reference_valid_only.csv
uploaded = files.upload()

print("Uploaded files:")
for name in uploaded.keys():
    print(name)

icd10_file = None

for name in uploaded.keys():
    lower = name.lower()

    if "processed_icd10_reference_valid_only" in lower or "icd10" in lower:
        icd10_file = name

print("Detected ICD-10 processed file:", icd10_file)

if icd10_file is None:
    raise ValueError("Could not detect processed_icd10_reference_valid_only.csv. Please upload the processed ICD-10 file.")

icd10_df = pd.read_csv(icd10_file)

print("icd10_df shape:", icd10_df.shape)
print("Columns:", icd10_df.columns.tolist())
display(icd10_df.head())

Saving processed_icd10_reference_valid_only (1).csv to processed_icd10_reference_valid_only (1).csv
Uploaded files:
processed_icd10_reference_valid_only (1).csv
Detected ICD-10 processed file: processed_icd10_reference_valid_only (1).csv
icd10_df shape: (9943, 10)
Columns: ['Code', 'Literal', 'Code_clean', 'Literal_basic', 'in_icd_reference', 'D_P', 'Description', 'y_category', 'y_full_code', 'Literal_match']


,Code,Literal,Code_clean,Literal_basic,in_icd_reference,D_P,Description,y_category,y_full_code,Literal_match
0,J9809,Hiperreactividad bronquial,J9809,Hiperreactividad bronquial,True,D,"Otras enfermedades de los bronquios, no clasif...",J,J9809,hiperreactividad bronquial
1,J9801,broncoespástica,J9801,broncoespástica,True,D,Broncoespasmo agudo,J,J9801,broncoespastica
2,I420,miocardiopatía dilatada,I420,miocardiopatía dilatada,True,D,Miocardiopatía dilatada,I,I420,miocardiopatia dilatada
3,Y831,HTA irc 6,Y831,HTA irc 6,True,D,Cirugía con implante de dispositivo interno ar...,Y,Y831,hta irc 6
4,R5600,Crisis febriles atípicas,R5600,Crisis febriles atípicas,True,D,Convulsiones febriles simples,R,R5600,crisis febriles atipicas


In [ ]:
#train-test split
category_counts = icd10_df["y_category"].value_counts()
rare_categories = category_counts[category_counts < 2].index

rare_df = icd10_df[icd10_df["y_category"].isin(rare_categories)].copy()
normal_df = icd10_df[~icd10_df["y_category"].isin(rare_categories)].copy()

train_normal, val_part = train_test_split(
    normal_df,
    test_size=0.2,
    random_state=42,
    stratify=normal_df["y_category"]
)

train_part = pd.concat([train_normal, rare_df], ignore_index=True)

train_codes = train_part["y_full_code"].astype(str).tolist()
train_categories = train_part["y_category"].astype(str).tolist()

print("Train shape:", train_part.shape)
print("Validation shape:", val_part.shape)
print("Train categories:", train_part["y_category"].nunique())
print("Validation categories:", val_part["y_category"].nunique())

Train shape: (7954, 10)
Validation shape: (1989, 10)
Train categories: 34
Validation categories: 33


In [ ]:
!pip install sentence-transformers

from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

train_texts = train_part["Literal_basic"].fillna("").astype(str).tolist()
val_texts = val_part["Literal_basic"].fillna("").astype(str).tolist()

train_embeddings = embed_model.encode(
    train_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

val_embeddings = embed_model.encode(
    val_texts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/125 [00:00<?, ?it/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

In [ ]:
#knn
knn = NearestNeighbors(n_neighbors=10, metric="cosine")
knn.fit(train_embeddings)

distances, indices = knn.kneighbors(val_embeddings)

top1_indices = indices[:, 0]

val_part["emb_ll_pred_code_top1"] = [train_codes[i] for i in top1_indices]
val_part["emb_ll_pred_category_top1"] = [train_categories[i] for i in top1_indices]
val_part["emb_ll_similarity_top1"] = 1 - distances[:, 0]

emb_ll_full_acc = accuracy_score(
    val_part["y_full_code"].astype(str),
    val_part["emb_ll_pred_code_top1"].astype(str)
)

emb_ll_cat_acc = accuracy_score(
    val_part["y_category"].astype(str),
    val_part["emb_ll_pred_category_top1"].astype(str)
)

print("Embedding literal-to-literal full-code accuracy:", emb_ll_full_acc)
print("Embedding literal-to-literal category accuracy:", emb_ll_cat_acc)

Embedding literal-to-literal full-code accuracy: 0.3815987933634992
Embedding literal-to-literal category accuracy: 0.6184012066365008


In [ ]:
#top k and weighted voting
knn = NearestNeighbors(n_neighbors=10, metric="cosine")
knn.fit(train_embeddings)

distances, indices = knn.kneighbors(val_embeddings)

top1_indices = indices[:, 0]

val_part["emb_ll_pred_code_top1"] = [train_codes[i] for i in top1_indices]
val_part["emb_ll_pred_category_top1"] = [train_categories[i] for i in top1_indices]
val_part["emb_ll_similarity_top1"] = 1 - distances[:, 0]

emb_ll_full_acc = accuracy_score(
    val_part["y_full_code"].astype(str),
    val_part["emb_ll_pred_code_top1"].astype(str)
)

emb_ll_cat_acc = accuracy_score(
    val_part["y_category"].astype(str),
    val_part["emb_ll_pred_category_top1"].astype(str)
)

print("Embedding literal-to-literal full-code accuracy:", emb_ll_full_acc)
print("Embedding literal-to-literal category accuracy:", emb_ll_cat_acc)

Embedding literal-to-literal full-code accuracy: 0.3815987933634992
Embedding literal-to-literal category accuracy: 0.6184012066365008


In [ ]:
#embedding centroid retrieval
train_embed_df = pd.DataFrame(train_embeddings)
train_embed_df["y_category"] = train_part["y_category"].astype(str).values

centroid_df = train_embed_df.groupby("y_category").mean(numeric_only=True).reset_index()

centroid_categories = centroid_df["y_category"].astype(str).tolist()
centroid_embeddings = centroid_df.drop(columns=["y_category"]).values

centroid_knn = NearestNeighbors(n_neighbors=5, metric="cosine")
centroid_knn.fit(centroid_embeddings)

centroid_distances, centroid_indices = centroid_knn.kneighbors(val_embeddings)

centroid_top1 = centroid_indices[:, 0]

val_part["emb_centroid_pred_category_top1"] = [
    centroid_categories[i] for i in centroid_top1
]

centroid_cat_acc = accuracy_score(
    val_part["y_category"].astype(str),
    val_part["emb_centroid_pred_category_top1"].astype(str)
)

print("Embedding category centroid accuracy:", centroid_cat_acc)

Embedding category centroid accuracy: 0.3273001508295626


In [ ]:
#tf idf retrieval
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def tfidf_retrieval_top1(train_texts, val_texts, train_codes, train_categories):
    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        lowercase=True,
        min_df=1
    )

    train_matrix = vectorizer.fit_transform(train_texts)
    val_matrix = vectorizer.transform(val_texts)

    sim = cosine_similarity(val_matrix, train_matrix)

    top1_idx = sim.argmax(axis=1)
    top1_scores = sim.max(axis=1)

    pred_codes = [train_codes[i] for i in top1_idx]
    pred_categories = [train_categories[i] for i in top1_idx]

    return pred_codes, pred_categories, top1_idx, top1_scores

In [ ]:
train_texts_ll = train_part["Literal_match"].fillna("").astype(str).tolist()
val_texts = val_part["Literal_match"].fillna("").astype(str).tolist()

tfidf_ll_codes, tfidf_ll_categories, tfidf_ll_indices, tfidf_ll_scores = tfidf_retrieval_top1(
    train_texts_ll,
    val_texts,
    train_codes,
    train_categories
)

tfidf_ll_cat_acc = accuracy_score(
    val_part["y_category"].astype(str),
    pd.Series(tfidf_ll_categories).astype(str)
)

print("TF-IDF literal-to-literal category accuracy:", tfidf_ll_cat_acc)

TF-IDF literal-to-literal category accuracy: 0.6963298139768728


In [ ]:
#BM25
!pip install rank-bm25

from rank_bm25 import BM25Okapi

def bm25_tokenize(text):
    return str(text).lower().split()

def bm25_retrieval_top1(train_texts, val_texts, train_codes, train_categories):
    tokenized_train = [bm25_tokenize(t) for t in train_texts]
    bm25 = BM25Okapi(tokenized_train)

    pred_codes = []
    pred_categories = []
    top1_scores = []

    for query in val_texts:
        scores = bm25.get_scores(bm25_tokenize(query))
        best_idx = int(np.argmax(scores))

        pred_codes.append(train_codes[best_idx])
        pred_categories.append(train_categories[best_idx])
        top1_scores.append(float(scores[best_idx]))

    return pred_codes, pred_categories, top1_scores

In [ ]:
bm25_codes, bm25_categories, bm25_scores = bm25_retrieval_top1(
    train_texts_ll,
    val_texts,
    train_codes,
    train_categories
)

bm25_cat_acc = accuracy_score(
    val_part["y_category"].astype(str),
    pd.Series(bm25_categories).astype(str)
)

print("BM25 literal-to-literal category accuracy:", bm25_cat_acc)

BM25 literal-to-literal category accuracy: 0.6470588235294118


In [ ]:
# ============================================================
# Save Retrieval Summary safely
# Works even if some optional experiments were not run
# ============================================================

summary_rows = []

# Embedding literal-to-literal
if "emb_ll_cat_acc" in globals():
    summary_rows.append({
        "method": "embedding_literal_top1",
        "category_accuracy": emb_ll_cat_acc
    })

# Embedding centroid
if "centroid_cat_acc" in globals():
    summary_rows.append({
        "method": "embedding_centroid_top1",
        "category_accuracy": centroid_cat_acc
    })

# TF-IDF literal retrieval
if "tfidf_ll_cat_acc" in globals():
    summary_rows.append({
        "method": "tfidf_literal_top1",
        "category_accuracy": tfidf_ll_cat_acc
    })

# BM25 literal retrieval
if "bm25_cat_acc" in globals():
    summary_rows.append({
        "method": "bm25_literal_top1",
        "category_accuracy": bm25_cat_acc
    })

retrieval_summary = pd.DataFrame(summary_rows)

# Add Top-K results only if that cell was run
if "topk_results_df" in globals():
    retrieval_summary = pd.concat(
        [retrieval_summary, topk_results_df],
        ignore_index=True
    )
else:
    print("topk_results_df not found. Skipping Top-K results.")

display(retrieval_summary.sort_values("category_accuracy", ascending=False))

retrieval_summary.to_csv("retrieval_experiment_summary.csv", index=False)
val_part.to_csv("retrieval_validation_predictions.csv", index=False)

print("Saved:")
print("retrieval_experiment_summary.csv")
print("retrieval_validation_predictions.csv")

topk_results_df not found. Skipping Top-K results.


,method,category_accuracy
2,tfidf_literal_top1,0.696330
3,bm25_literal_top1,0.647059
0,embedding_literal_top1,0.618401
1,embedding_centroid_top1,0.327300


Saved:
retrieval_experiment_summary.csv
retrieval_validation_predictions.csv
